# توقع أسعار البيوت: Linear Regression من الصفر بـ NumPy
### Gradient Descent vs Adam — ونفس الداتا بثلاث أسئلة

**الخطة:**
1. نضيف داتا من Kaggle للـ Notebook
2. نفهمها ونرسمها
3. Linear Regression بـ **Gradient Descent** (من الصفر)
4. نفس الموديل بـ **Adam**
5. نتحقق من الجواب بالـ Normal Equation
6. **Polynomial Regression**: هل العلاقة منحنية؟
7. **Logistic Regression**: هل البيت غالي؟

> القاعدة: كل الرياضيات بـ NumPy فقط. ما في scikit-learn ولا pandas. الـ matplotlib للرسم بس.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4.5)

## 1. الداتا: Housing Prices Dataset
**Dataset:** `Housing Prices Dataset` من حساب **yasserh**
الرابط: kaggle.com/datasets/yasserh/housing-prices-dataset
545 بيت، والأعمدة المهمة إلنا: `area` (المساحة) و `price` (السعر).

**كيف نضيفها للـ Notebook على Kaggle:**
1. من يمين الشاشة: **Add Input** (أو Add Data)
2. ابحث عن `Housing Prices Dataset` واختار اللي صاحبها yasserh
3. اضغط **+**، بتنحط الملفات تحت `/kaggle/input/`

الكود تحت بيدوّر على `Housing.csv` لحاله. إذا ما لقاه، بيولّد داتا شبيهة عشان الجلسة ما توقف.

In [ ]:
import glob

X_COL = "area"    # feature: house area (sq ft)
Y_COL = "price"   # target: house price

def load_kaggle():
    files = glob.glob("/kaggle/input/**/Housing.csv", recursive=True)
    if not files:
        raise FileNotFoundError("Housing.csv not found -> did you click 'Add Input'?")
    print("Loaded:", files[0])
    # dtype=float: text columns (mainroad, ...) become nan, we don't use them
    data = np.genfromtxt(files[0], delimiter=",", names=True, dtype=float, encoding="utf-8")
    print("Columns:", data.dtype.names)
    return data[X_COL], data[Y_COL]

try:
    x_raw, y = load_kaggle()
except Exception as e:
    print("Using synthetic backup data ->", e)
    x_raw = np.random.uniform(1650, 12000, 545)
    y = np.clip(2.4e6 + 460 * x_raw + np.random.normal(0, 1.4e6, 545), 1.75e6, None)

mask = ~np.isnan(x_raw) & ~np.isnan(y)   # drop missing values
x_raw, y = x_raw[mask], y[mask] / 1e6    # price in MILLIONS (see note below)
m = len(y)
print(f"m = {m} houses")
print(f"area: {x_raw.min():.0f} -> {x_raw.max():.0f} | price: {y.min():.2f}M -> {y.max():.2f}M")

> **ليش قسمنا السعر على مليون؟** الأسعار بالملايين، فالـ MSE بيطلع أرقام بحجم $10^{12}$ وصعب نقرأها، والأهم إن Adam خطوته تقريباً بحجم الـ lr، فلو الـ bias الصحيح 4,000,000 بيحتاج ملايين الخطوات. تغيير الوحدة ما بيغيّر العلاقة أبداً، بس بيخلي الأرقام "إنسانية".

## 2. نشوف الداتا
قبل أي موديل: **ارسم**. الرسمة بتحكيلك أي موديل منطقي.

In [ ]:
plt.scatter(x_raw, y, alpha=0.6)
plt.xlabel("Area (sq ft)"); plt.ylabel("Price (millions)"); plt.title("Raw data")
plt.show()

## 3. الموديل والـ Loss

$$\hat{y} = w x + b$$

$$J(w,b) = \frac{1}{m}\sum_{i=1}^{m}(\hat{y}_i - y_i)^2 \quad \text{(MSE)}$$

**المشتقات (Gradients):**
$$\frac{\partial J}{\partial w} = \frac{2}{m}\sum(\hat{y}_i - y_i)\,x_i \qquad \frac{\partial J}{\partial b} = \frac{2}{m}\sum(\hat{y}_i - y_i)$$

### Feature Scaling
بنحوّل x بحيث يصير متوسطها 0 وانحرافها 1. بدون هذا الخطوة، GD بيكون بطيء أو بيتذبذب.

In [ ]:
mu, sigma = x_raw.mean(), x_raw.std()
x = (x_raw - mu) / sigma          # standardized feature

def predict(x, w, b):
    return w * x + b

def mse(y_hat, y):
    return np.mean((y_hat - y) ** 2)

def gradients(x, y, w, b):
    error = predict(x, w, b) - y
    dw = 2 * np.mean(error * x)
    db = 2 * np.mean(error)
    return dw, db

## 4. Gradient Descent
الفكرة: انت واقف على جبل بالضباب وبدك تنزل للوادي. بتحس الميل تحت رجليك (gradient) وبتمشي بعكسه خطوة بحجم **learning rate**.

$$w \leftarrow w - \alpha \frac{\partial J}{\partial w} \qquad b \leftarrow b - \alpha \frac{\partial J}{\partial b}$$

In [ ]:
def gradient_descent(x, y, lr=0.1, epochs=100):
    w, b = 0.0, 0.0
    history = []
    for epoch in range(epochs):
        dw, db = gradients(x, y, w, b)
        w -= lr * dw
        b -= lr * db
        history.append(mse(predict(x, w, b), y))
    return w, b, history

w_gd, b_gd, hist_gd = gradient_descent(x, y, lr=0.1, epochs=200)
print(f"GD  -> w = {w_gd:.3f}, b = {b_gd:.3f}, final MSE = {hist_gd[-1]:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(hist_gd); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("MSE"); ax[0].set_title("Loss curve (GD)")
order = np.argsort(x_raw)
ax[1].scatter(x_raw, y, alpha=0.5)
ax[1].plot(x_raw[order], predict(x[order], w_gd, b_gd), "r", lw=2)
ax[1].set_xlabel("Area (sq ft)"); ax[1].set_ylabel("Price (millions)"); ax[1].set_title("Fitted line")
plt.show()

### تجربة حيّة: أثر الـ Learning Rate
- صغير جداً ← بطيء
- مناسب ← بينزل بسرعة ويستقر
- كبير جداً ← **بينفجر** (diverge)

In [ ]:
for lr in [0.01, 0.1, 0.9, 1.05]:
    _, _, h = gradient_descent(x, y, lr=lr, epochs=30)
    plt.plot(h, label=f"lr = {lr}")
plt.yscale("log"); plt.xlabel("Epoch"); plt.ylabel("MSE (log scale)")
plt.title("Effect of learning rate"); plt.legend(); plt.show()

## 5. Adam
مشكلة GD العادي: نفس حجم الخطوة لكل parameter وبكل وقت. Adam بيحل هالشي بفكرتين:

1. **Momentum** ($m$): متوسط متحرك للـ gradients، كأنه كرة بتتدحرج وبتاخذ زخم بالاتجاه الصح.
2. **Adaptive step** ($v$): متوسط متحرك لمربع الـ gradients؛ الـ parameter اللي gradient تبعه كبير بياخذ خطوة أصغر والعكس.

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$$
$$\hat m_t = \frac{m_t}{1-\beta_1^t} \qquad \hat v_t = \frac{v_t}{1-\beta_2^t} \qquad \theta \leftarrow \theta - \alpha\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}$$

الـ bias correction ($\hat m, \hat v$) لأن m و v بيبدوا من صفر، فبالبداية بيكونوا أصغر من الحقيقة.

In [ ]:
def adam(x, y, lr=0.1, epochs=200, beta1=0.9, beta2=0.999, eps=1e-8):
    theta = np.zeros(2)            # [w, b]
    m_t = np.zeros(2)              # 1st moment (momentum)
    v_t = np.zeros(2)              # 2nd moment (squared gradients)
    history = []
    for t in range(1, epochs + 1):
        g = np.array(gradients(x, y, theta[0], theta[1]))
        m_t = beta1 * m_t + (1 - beta1) * g
        v_t = beta2 * v_t + (1 - beta2) * g**2
        m_hat = m_t / (1 - beta1**t)
        v_hat = v_t / (1 - beta2**t)
        theta -= lr * m_hat / (np.sqrt(v_hat) + eps)
        history.append(mse(predict(x, theta[0], theta[1]), y))
    return theta[0], theta[1], history

w_adam, b_adam, hist_adam = adam(x, y, lr=0.1, epochs=200)
print(f"Adam -> w = {w_adam:.3f}, b = {b_adam:.3f}, final MSE = {hist_adam[-1]:.3f}")

In [ ]:
plt.plot(hist_gd, label="Gradient Descent (lr=0.1)")
plt.plot(hist_adam, label="Adam (lr=0.1)")
plt.yscale("log"); plt.xlabel("Epoch"); plt.ylabel("MSE (log scale)")
plt.title("GD vs Adam"); plt.legend(); plt.show()

> **ملاحظة للنقاش:** بمسألة سهلة زي هاي (parameter اثنين، loss شكلها وعاء)، GD المضبوط ممكن يكون منافس جداً. قوة Adam بتبان بالشبكات العصبية اللي فيها ملايين parameters والـ loss surface معقد. **لاحظوا التذبذب** بمنحنى Adam: هذا أثر الـ momentum.
>
> **نقطة دقيقة:** خطوة Adam تقريباً بحجم الـ lr مهما كان الـ gradient كبير (لأننا بنقسم على $\sqrt{\hat v}$). فلو الـ bias الصحيح ≈ متوسط السعر (حوالي 4.8) و lr = 0.01، بدنا مئات الخطوات لنوصل. جرّبوا `lr=0.01` وشوفوا الفرق.

## 6. التحقق: Normal Equation
Linear Regression إلها حل مباشر بدون تكرار:
$$\theta = (X^TX)^{-1}X^Ty$$
إذا GD و Adam صح، لازم يطلعوا قريبين من هالجواب.

**سؤال للطلاب:** إذا في حل مباشر، ليش بنستخدم GD أصلاً؟

In [ ]:
X_mat = np.column_stack([x, np.ones(m)])
theta_exact = np.linalg.solve(X_mat.T @ X_mat, X_mat.T @ y)
print(f"Exact -> w = {theta_exact[0]:.3f}, b = {theta_exact[1]:.3f}")
print(f"GD    -> w = {w_gd:.3f}, b = {b_gd:.3f}")
print(f"Adam  -> w = {w_adam:.3f}, b = {b_adam:.3f}")

# back to original units: price = w_orig * area + b_orig
w_orig = theta_exact[0] / sigma
b_orig = theta_exact[1] - theta_exact[0] * mu / sigma
print(f"\nIn real units: price = {w_orig*1e6:.1f} * area + {b_orig*1e6:,.0f}")
print(f"Each extra 100 sq ft adds about {w_orig*1e8:,.0f} to the price")

---
# نفس الداتا... أسئلة مختلفة

| الموديل | السؤال |
|---|---|
| Linear | كم سعر البيت؟ |
| Polynomial | هل كل متر إضافي بيرفع السعر بنفس القدر، ولا الزيادة بتقل؟ |
| Logistic | هل البيت غالي (أعلى من الوسيط)؟ |

## 7. Polynomial Regression
الحيلة: ما بنغيّر الخوارزمية أبداً. بس بنضيف feature جديدة $x^2$:
$$\hat y = w_1 x + w_2 x^2 + b$$
هي لسا **linear بالـ parameters**، فنفس GD بيشتغل. بنعمّم الكود ليشتغل على matrix:

In [ ]:
def gd_matrix(X, y, lr=0.1, epochs=500):
    # X: (m, n) features ; returns weights (n,), bias, history
    w, b = np.zeros(X.shape[1]), 0.0
    history = []
    for _ in range(epochs):
        error = X @ w + b - y
        w -= lr * 2 * X.T @ error / len(y)
        b -= lr * 2 * np.mean(error)
        history.append(np.mean((X @ w + b - y) ** 2))
    return w, b, history

x2 = x_raw ** 2
X_poly = np.column_stack([x, (x2 - x2.mean()) / x2.std()])   # scale x^2 too

w_poly, b_poly, hist_poly = gd_matrix(X_poly, y, lr=0.1, epochs=2000)
w_lin,  b_lin,  hist_lin  = gd_matrix(x.reshape(-1, 1), y, lr=0.1, epochs=2000)
print(f"Linear     MSE = {hist_lin[-1]:.3f}")
print(f"Polynomial MSE = {hist_poly[-1]:.3f}")

In [ ]:
order = np.argsort(x_raw)
plt.scatter(x_raw, y, alpha=0.4)
plt.plot(x_raw[order], (x.reshape(-1,1) @ w_lin + b_lin)[order], "r", lw=2, label="Linear")
plt.plot(x_raw[order], (X_poly @ w_poly + b_poly)[order], "g", lw=2, label="Polynomial (deg 2)")
plt.xlabel("Area (sq ft)"); plt.ylabel("Price (millions)"); plt.legend(); plt.show()

> **سؤال للطلاب:** إذا الـ MSE قلّ شوي بس، هل يستاهل نعقّد الموديل؟ وإذا رفعنا الـ degree لـ 15، شو رح يصير؟ (← overfitting ← Ridge/Lasso)

## 8. Logistic Regression
السؤال تغيّر: مش "كم السعر؟" بل "**البيت غالي ولا لا؟**". الـ target صار 0 أو 1 (غالي = أعلى من الـ median).

ليش ما نستخدم Linear؟ لأنها ممكن تطلع 1.7 أو -0.4، وهذا مش احتمال. الحل: نمرر الناتج على **sigmoid**:
$$\sigma(z) = \frac{1}{1+e^{-z}} \qquad \hat p = \sigma(wx+b)$$

والـ loss صار **Binary Cross-Entropy** بدل MSE:
$$J = -\frac{1}{m}\sum\big[y\log\hat p + (1-y)\log(1-\hat p)\big]$$

المفاجأة الحلوة: الـ gradient إله **نفس شكل** Linear Regression تقريباً: $\frac{1}{m}\sum(\hat p - y)x$

In [ ]:
THRESHOLD = np.median(y)                      # "expensive" = above median price
y_cls = (y > THRESHOLD).astype(float)
print(f"Threshold = {THRESHOLD:.2f}M | expensive: {int(y_cls.sum())}, not: {int(m - y_cls.sum())}")

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def bce(p, y, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def logistic_gd(x, y, lr=0.5, epochs=1000):
    w, b = 0.0, 0.0
    history = []
    for _ in range(epochs):
        p = sigmoid(w * x + b)
        w -= lr * np.mean((p - y) * x)
        b -= lr * np.mean(p - y)
        history.append(bce(sigmoid(w * x + b), y))
    return w, b, history

w_log, b_log, hist_log = logistic_gd(x, y_cls)
p = sigmoid(w_log * x + b_log)
acc = np.mean((p >= 0.5) == y_cls)
print(f"Final BCE = {hist_log[-1]:.3f} | Accuracy = {acc:.1%}")

In [ ]:
grid = np.linspace(x_raw.min(), x_raw.max(), 200)
p_grid = sigmoid(w_log * (grid - mu) / sigma + b_log)
boundary = mu - b_log * sigma / w_log          # where p = 0.5

plt.scatter(x_raw, y_cls, alpha=0.4, label="Actual (0 = normal, 1 = expensive)")
plt.plot(grid, p_grid, "purple", lw=2, label="P(expensive)")
plt.axvline(boundary, ls="--", c="gray", label=f"Decision boundary ~ {boundary:.0f} sq ft")
plt.xlabel("Area (sq ft)"); plt.ylabel("P(expensive)"); plt.legend(); plt.show()

for a in [3000, 6000, 9000]:
    pa = sigmoid(w_log * (a - mu) / sigma + b_log)
    print(f"{a} sq ft -> P(expensive) = {pa:.2f}")

---
## الخلاصة
| | Linear | Polynomial | Logistic |
|---|---|---|---|
| السؤال | كم؟ | كم؟ (علاقة منحنية) | نعم/لا؟ |
| الناتج | رقم | رقم | احتمال 0→1 |
| Loss | MSE | MSE | Binary Cross-Entropy |
| التعديل على Linear | — | features جديدة ($x^2$) | sigmoid على الناتج |

**أهم فكرة:** كلهم نفس العائلة، ونفس الـ optimizer (GD أو Adam) بيدرّبهم. الشبكة العصبية = logistic units كثيرة فوق بعض.

## تمارين
1. جرّب Adam بـ `lr=0.01` و `lr=1`. شو بتلاحظ؟
2. شيل الـ feature scaling وشغّل GD. شو صار؟
3. ارفع degree الـ Polynomial لـ 3. هل تحسّن؟
4. خلّي `THRESHOLD` يساوي `np.percentile(y, 75)`. كيف تحرّك الـ decision boundary؟
5. **تحدي:** استخدم `bedrooms` و `bathrooms` مع `area` (Multiple Regression) بدالة `gd_matrix`.
6. **تحدي:** طبّق Adam على Logistic Regression.